# BioKEA OpenSCAD CAD custody — independent replay

This notebook verifies the exact-byte OpenSCAD outputs and every MMR prefix. This is a CAD software run, **not** a robotics or biological experiment. Source CAD dimensions are provisional MEASURE placeholders.


In [1]:
import sys, json, hashlib, pathlib, subprocess, tempfile
from pathlib import Path
P=Path.cwd()
assert (P/'cad_build.py').exists(), 'Run notebook from experiment directory'
sys.path.insert(0,str(P))
import cad_build as b
print('OpenSCAD:', subprocess.run(['openscad','--version'],capture_output=True,text=True).stderr.strip())

OpenSCAD: OpenSCAD version 2026.10.11


## Identity, ordered MMR prefixes and independent calculation

In [2]:
report=b.verify(); print(json.dumps(report,indent=2)); assert report['PASS']
rows=b.read_rows(); assert len(rows)==report['leaves']
for i,r in enumerate(rows):
    assert b.digest(b.canon(r['record']))==r['event_hash']
    p=json.loads((b.OUT/f'prefix-{i+1:06d}.json').read_text())
    assert p['root']==b.mmr(rows[:i+1])['root']==b.independent_root(rows[:i+1])
print('ALL PREFIXES PASS')


{
  "PASS": true,
  "algorithm": "HYDRALAMP_MMR_V1",
  "leaves": 7,
  "root": "39b126f696333d2eb120bfc7f62c3a1cc2577064d717ab3c51268f1c58e27504",
  "peaks": [
    "3fd58751726333f104b2395b4bac84edda5a2e513e4ce7db9f61932c0afba00b",
    "fb60d5c5d6faa7a0ef11dd3535f4b6b600cdc0c7e7a65ee7048d8827794d10aa",
    "076739a2602264cebc55b62a4d6f1ba274c6920c494e72c4d24a0e592ad16dc4"
  ],
  "identity_recomputed": true,
  "prefixes_recomputed": true,
  "independent_root_recomputed": true,
  "signing": "NOT_SIGNED",
  "external_antigense_mmr_updated": false
}
ALL PREFIXES PASS


## CAD exact-byte replay — read-only source, temporary outputs

In [3]:
src=P/'tower_three_analogs.scad'
with tempfile.TemporaryDirectory() as tmp:
    t=Path(tmp)
    for suffix,cmd_args,published in [('png',['--imgsize=1200,960','--autocenter','--viewall'],b.OUT/'tower_preview.png'),('stl',[],b.OUT/'tower_mesh.stl')]:
        path=t/('tower.'+suffix)
        proc=subprocess.run(['openscad','-o',str(path),*cmd_args,str(src)],capture_output=True,text=True,timeout=160)
        assert proc.returncode==0,(suffix,proc.stderr)
        identical=b.hash_file(path)==b.hash_file(published)
        print(suffix, 'EXACT_BYTES_PASS' if identical else 'MISMATCH',b.hash_file(path))
        assert identical


png EXACT_BYTES_PASS c28c1303f9c211ab3a1e4afa24303f8e057e50ffc92b9eb3c691a73ca79a7463
stl EXACT_BYTES_PASS 99b1e5a951b7dfc3d35f484a9d56bf8b82b27b1c28c91e451a0ed5a303722279


## Independent Antigense CyberHack algorithm comparison

In [4]:
import importlib.util
ref=Path('/tmp/antigense_cad_reference_20261025/src/custody.py')
if ref.exists():
    spec=importlib.util.spec_from_file_location('antigense_upstream',ref)
    mod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)
    assert mod.mmr(rows)['root']==b.mmr(rows)['root']
    print('ANTIGENSE REFERENCE PARITY PASS; original custody unchanged')
else:
    print('NOT_TESTED: independently clone upstream Antigense source for parity check')


ANTIGENSE REFERENCE PARITY PASS; original custody unchanged


## Negative test: data mutation in memory, without touching the ledger

In [5]:
import copy
mut=copy.deepcopy(rows)
mut[0]['record']['kind']='TAMPER_NEGATIVE_TEST'
v=b.verify(mut)
print('TAMPER_REJECTED',not v['PASS'],v)
assert v['PASS'] is False
print('CLAIM LIMIT: hashes establish bytes and ordering; physical motion, specimen recovery and identity remain NOT_TESTED')


TAMPER_REJECTED True {'PASS': False, 'failure': 'event identity mismatch', 'at': 1}
CLAIM LIMIT: hashes establish bytes and ordering; physical motion, specimen recovery and identity remain NOT_TESTED
